# 09. Prediction Pipeline

This notebook builds a simple prediction pipeline for future groundwater level estimation.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

## Load Feature-Engineered Dataset

In [ ]:
candidate_paths = [
    Path("../datasets/groundwater_feature_engineered.csv"),
    Path("datasets/groundwater_feature_engineered.csv"),
    Path("groundwater_feature_engineered.csv")
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("groundwater_feature_engineered.csv not found in expected paths.")

df = pd.read_csv(dataset_path, parse_dates=["Data Acquisition Time"])

print(f"Dataset path: {dataset_path.resolve()}")
print("Shape:", df.shape)

## Define Features and Chronological Split

The same feature list and 80-20 split are used for consistency.

In [ ]:
target = "Groundwater Level Telemetry 6 Hourly (meter)"

features = [
    "Latitude", "Longitude", "RL_MSL",
    "Year", "Month", "Day", "Hour",
    "DayOfWeek", "WeekOfYear", "Quarter", "IsWeekend",
    "Lag_1", "Lag_4", "Lag_28",
    "RollingMean_4", "RollingStd_4",
    "Hour_sin", "Hour_cos", "Month_sin", "Month_cos",
    "Station_ID"
]

split_index = int(len(df) * 0.80)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

## Load Best Model if Available, Otherwise Train and Save

Model files are stored in the `models` folder for reuse.

In [ ]:
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

best_model_path = model_dir / "best_model.pkl"
meta_path = model_dir / "best_model_meta.json"

def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²": r2_score(y_true, y_pred)
    }

if best_model_path.exists() and meta_path.exists():
    best_model = joblib.load(best_model_path)
    model_meta = json.loads(meta_path.read_text())
    best_model_name = model_meta.get("model_name", "Saved Model")
    print(f"Loaded saved best model: {best_model_name}")
else:
    candidates = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
        "XGBoost": XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    }

    rows = []
    fitted_models = {}

    for name, model in candidates.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        m = evaluate(y_test, pred)
        rows.append({"Model": name, **m})
        fitted_models[name] = model

    model_report = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
    display(model_report.style.format({"MAE": "{:.4f}", "RMSE": "{:.4f}", "R²": "{:.4f}"}))

    best_model_name = model_report.loc[0, "Model"]
    best_model = fitted_models[best_model_name]

    joblib.dump(best_model, best_model_path)
    meta_path.write_text(json.dumps({"model_name": best_model_name, "features": features}, indent=2))

    print(f"Trained and saved best model: {best_model_name}")

# quick sanity check on current test split
test_pred = best_model.predict(X_test)
metrics = evaluate(y_test, test_pred)
print("\nCurrent model performance on test split:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

## Reusable Prediction Function

In [ ]:
def predict_groundwater(input_df, model, feature_columns):
    """Predict groundwater level from a dataframe containing model features."""

    missing_cols = [col for col in feature_columns if col not in input_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    model_input = input_df[feature_columns].copy()
    preds = model.predict(model_input)

    output = input_df.copy()
    output["Predicted_Groundwater_Level_m"] = preds
    return output

## Example Predictions (Sample Rows from Dataset)

In [ ]:
sample_input = X_test.head(10).copy()
prediction_output = predict_groundwater(sample_input, best_model, features)
prediction_output.head()

In [ ]:
comparison_output = pd.DataFrame({
    "Actual": y_test.head(10).values,
    "Predicted": prediction_output["Predicted_Groundwater_Level_m"].values
})
comparison_output["Absolute_Error"] = np.abs(comparison_output["Actual"] - comparison_output["Predicted"])
comparison_output

## Single Input Example

This format can be used later when new feature rows are prepared from incoming telemetry data.

In [ ]:
single_input = X_test.head(1).copy()
single_pred = predict_groundwater(single_input, best_model, features)
single_pred[["Predicted_Groundwater_Level_m"]]

## Conclusion

The prediction pipeline is now ready for reuse.
For future forecasts, prepare new rows with the same 21 engineered features and pass them to the `predict_groundwater` function.